# RAG 기반 금융 QA & 서비스 고려사항

## 1. RAG 구조 이해

### RAG(Retrieval-Augmented Generation)란?

LLM은 학습 시점 이후의 정보나, 특정 회사/문서에만 있는 비공개 정보를 알지 못한다.

**RAG**는 질문이 들어오면 관련된 문서를 먼저 "검색(Retrieval)"해서 찾아온 뒤,
그 내용을 LLM에게 참고자료로 넘겨 답변을 "생성(Generation)"하게 하는 구조다.

**처리 흐름**

문서 준비 → 조각내기(Chunking) → 임베딩(Embedding) → 벡터DB 저장
→ 질문 입력 → 유사 조각 검색(Retrieval) → LLM에 전달 → 답변 생성

이렇게 하면 LLM이 모르는 내용도, 검색된 문서를 근거로 답할 수 있고(환각 감소),
답변의 **출처**를 함께 제시할 수 있다는 장점이 있다.

---

### 0. 환경설정

In [38]:
# !pip install langchain langchain_community langchain_text_splitters chromadb langchain_openai openai -q
# !pip install PyPDF2 -q
# !pip install -U langchain-classic -q
# !pip install rank_bm25 -q

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI

from langchain_classic.chains import create_retrieval_chain, RetrievalQA
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

from openai import OpenAI

import PyPDF2
import sys, os
import re

os.environ['OPENAI_API_KEY'] = ''


### 2. 문서 조각내기 (Chunking)

`RecursiveCharacterTextSplitter`로 긴 문서를 작은 조각(chunk)으로 나눈다.

- **chunk_size**: 조각 하나의 최대 길이
  - 너무 작으면(예: 30) 문장이 중간에 잘려 정보가 부족해짐
  - 너무 크면(예: 200) 문맥은 풍부해지지만 불필요한 내용까지 섞임
- **chunk_overlap**: 다음 조각을 만들 때 이전 조각의 끝부분을 얼마나 반복해서 포함시킬지
  - overlap이 있으면 조각 경계에서 문맥이 끊기지 않고 자연스럽게 이어짐
  - **주의**: 첫 번째 조각은 "이전 조각"이 없기 때문에 overlap 값과 무관하게 항상 동일하게 만들어진다. overlap 효과는 두 번째 조각부터 확인해야 한다.

In [9]:
# 실습 1

# 방법 1 : 직접 입력
# report = '''B전자 3분기 매출 70조원(젼년 대비 +12%),
# 영업이익 10조원(+25%), 메모리 수요 회복이 주효.
# 4분기 가이던스는 보수적으로 제시했다.'''

# 방법 2 : TEXT 파일 읽기
# with open("report.txt", 'r', encoding="utf-8") as f:
#   report = f.read()

# 방법 3 : PDF 파일 읽기
def extract_pdf_text(path):
    text = ''
    with open(path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ''
    return text
report = extract_pdf_text("economy_news_sample.pdf")


splitter = RecursiveCharacterTextSplitter(
    chunk_size = 50, chunk_overlap=10)     # 문맥당 50개를 10개씩 겹쳐서 조각 가져옴

docs = splitter.create_documents([report])

print('조각 수 : ', len(docs))

# 저장된 조각 내용 확인
for i, doc in enumerate(docs,1):
    print(f"\n[{i}번째 조각]")
    print(doc.page_content)

조각 수 :  20

[1번째 조각]
경제 뉴스 실습 자료
 금융 텍스트 분석 수업용 예시 기사 - 2026년 7월

[2번째 조각]
1. 반도체 수출 증가, 제조업 회복 기대

[3번째 조각]
국내 반도체 기업의 고대역폭 메모리와 인공지능용 부품 수출이 늘면서 제조업 경기 회복에

[4번째 조각]
경기 회복에 대한 기대가

[5번째 조각]
커지고 있다. 업계는 하반기에도 데이터센터 투자가 이어질 경우 관련 기업의 매출과

[6번째 조각]
기업의 매출과 영업이익이 개선될

[7번째 조각]
것으로 전망했다.
2. 원자재 가격 상승, 중소 제조업 비용 부담

[8번째 조각]
국제 원자재 가격과 운송비가 오르면서 중소 제조업체의 생산비 부담이 확대되고 있다. 일부

[9번째 조각]
있다. 일부 기업은 제품 가격

[10번째 조각]
조정을 검토하고 있지만 소비 위축 가능성 때문에 즉각적인 인상에는 신중한 모습이다.

[11번째 조각]
3. 기준금리 동결, 금융시장 변동성은 지속

[12번째 조각]
기준금리가 동결되면서 대출 이자 부담의 급격한 증가는 제한될 것으로 보인다. 다만 환율과

[13번째 조각]
다만 환율과 해외 금리

[14번째 조각]
움직임에 따라 주식과 채권시장의 변동성이 이어질 수 있어 투자자는 기업 실적과 재무

[15번째 조각]
기업 실적과 재무 건전성을 함께 확인할

[16번째 조각]
필요가 있다.
 기사
 감성 예시
 핵심 키워드
 
반도체 수출 증가
긍정

[17번째 조각]
긍정
수출, AI 반도체, 실적 개선
원자재 가격 상승
부정
원가, 운송비, 가격 인상

[18번째 조각]
기준금리 동결
중립
금리, 환율, 시장 변동성

[19번째 조각]
※ 이 문서는 실제 신문 기사를 복사한 것이 아니라 수업과 코드 실습을 위해 작성한 예시

[20번째 조각]
위해 작성한 예시 자료입니다.


### 3. 임베딩 & 벡터 DB

- **임베딩(Embedding)**: 텍스트를 의미를 담은 숫자 벡터로 변환하는 것 (`OpenAIEmbeddings`)
- **벡터 DB(Chroma)**: 임베딩된 조각들을 저장해두고, 질문 벡터와 가장 가까운(유사한) 조각을 빠르게 찾아주는 저장소
- `Chroma.from_documents(docs, OpenAIEmbeddings())`로 생성
- 같은 컬렉션에 데이터가 계속 쌓이는 걸 막으려면 `collection_name`을 지정하거나 `delete_collection()`으로 초기화

In [11]:
# 실습 2 앞 실습의 docs를 임베딩해 DB 생성

# 누적된 것을 초기화하는 코드

try :
  db.delete_collection()
except :
  pass

db = Chroma.from_documents(docs, OpenAIEmbeddings())


print('저장된 조각 수 : ', db._collection.count())

data = db.get()

for i, text in enumerate(data['documents'],1):
    print(f"\n[{i}번째 조각]")
    print(text)

# 설정으로 인해 글자수들이 잘리는 현상 발생

저장된 조각 수 :  20

[1번째 조각]
경제 뉴스 실습 자료
 금융 텍스트 분석 수업용 예시 기사 - 2026년 7월

[2번째 조각]
1. 반도체 수출 증가, 제조업 회복 기대

[3번째 조각]
국내 반도체 기업의 고대역폭 메모리와 인공지능용 부품 수출이 늘면서 제조업 경기 회복에

[4번째 조각]
경기 회복에 대한 기대가

[5번째 조각]
커지고 있다. 업계는 하반기에도 데이터센터 투자가 이어질 경우 관련 기업의 매출과

[6번째 조각]
기업의 매출과 영업이익이 개선될

[7번째 조각]
것으로 전망했다.
2. 원자재 가격 상승, 중소 제조업 비용 부담

[8번째 조각]
국제 원자재 가격과 운송비가 오르면서 중소 제조업체의 생산비 부담이 확대되고 있다. 일부

[9번째 조각]
있다. 일부 기업은 제품 가격

[10번째 조각]
조정을 검토하고 있지만 소비 위축 가능성 때문에 즉각적인 인상에는 신중한 모습이다.

[11번째 조각]
3. 기준금리 동결, 금융시장 변동성은 지속

[12번째 조각]
기준금리가 동결되면서 대출 이자 부담의 급격한 증가는 제한될 것으로 보인다. 다만 환율과

[13번째 조각]
다만 환율과 해외 금리

[14번째 조각]
움직임에 따라 주식과 채권시장의 변동성이 이어질 수 있어 투자자는 기업 실적과 재무

[15번째 조각]
기업 실적과 재무 건전성을 함께 확인할

[16번째 조각]
필요가 있다.
 기사
 감성 예시
 핵심 키워드
 
반도체 수출 증가
긍정

[17번째 조각]
긍정
수출, AI 반도체, 실적 개선
원자재 가격 상승
부정
원가, 운송비, 가격 인상

[18번째 조각]
기준금리 동결
중립
금리, 환율, 시장 변동성

[19번째 조각]
※ 이 문서는 실제 신문 기사를 복사한 것이 아니라 수업과 코드 실습을 위해 작성한 예시

[20번째 조각]
위해 작성한 예시 자료입니다.


In [12]:
# 실습 3 벡터 DB에서 질문과 유사한 문서 조각을 검색하시오

query = '핵심 키워드는 뭐야?'
# query = '반도체 전망은 어때?'

hits = db.similarity_search(query, k=2) # 질문, 출력 개수 조정

for i,h in enumerate(hits):
  print(f'[{i+1}]', h.page_content)

[1] 필요가 있다.
 기사
 감성 예시
 핵심 키워드
 
반도체 수출 증가
긍정
[2] 위해 작성한 예시 자료입니다.


## 2. 금융문서 기반 QA 챗봇 구현



### 4. 검색기 (Retriever)

- `db.as_retriever(search_kwargs={'k': 3})` → 벡터DB를 "질문에 대해 상위 k개 조각을 찾아주는 객체"로 변환
- `db.similarity_search(query, k=n)`로 직접 검색 테스트 가능

### 5. QA 체인 구성 방식 두 가지

#### 방식 A. `RetrievalQA` (레거시, deprecated)

```python
qa = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model_name='gpt-4o-mini'),
    retriever=retriever,
    return_source_documents=True)

res = qa.invoke({'query': '질문'})
res['result']              # 답변
res['source_documents']    # 출처 조각 리스트
```

- langchain 1.0 이후 `langchain.chains`에서 빠지고 `langchain_classic.chains`로 이동됨

#### 방식 B. `create_retrieval_chain` (최신, 권장)

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 참고하여 질문에 답하세요.\n\n{context}"),
    ("human", "{input}")
])
documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
qa = create_retrieval_chain(retriever, documents_chain)

res = qa.invoke({'input': '질문'})
res['answer']    # 답변
res['context']   # 출처 조각 리스트
```

- 프롬프트를 직접 작성하기 때문에 "모르면 모른다고 답하라" 같은 지시를 넣기 쉬움
- 키 이름이 다름: `query`→`input`, `result`→`answer`, `source_documents`→`context`

In [14]:
# 실습 1 벡터 DB를 검색기로 만들어 QA 체인을 구성하시오

qa = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model_name='gpt-4o-mini'),
    retriever=db.as_retriever(search_kwargs={'k':3}),
    return_source_documents=True)

print('QA 챗봇 준비 완료')

QA 챗봇 준비 완료


In [25]:
# 실습 2 QA 챗봇에 질문하고 답변과 출처를 함께 출력하시오

result = qa.invoke({'query' : '핵심 키워드는?'})

print('[답변]', result['result'])
print('[출처]', result['source_documents'][0].page_content[:40])

[답변] 핵심 키워드는 "반도체 수출 증가", "기준금리 동결", "금리", "환율", "시장 변동성"입니다.
[출처] [Document(metadata={}, page_content='필요가 있다.\n 기사\n 감성 예시\n 핵심 키워드\n \n반도체 수출 증가\n긍정'), Document(metadata={}, page_content='위해 작성한 예시 자료입니다.'), Document(metadata={}, page_content='기준금리 동결\n중립\n금리, 환율, 시장 변동성')]


In [23]:
# 실습 3 자료에 없는 내용은 '모른다'고 답변하는지 확인

result = qa.invoke({'query' : '경쟁사 D전자의 순이익은?'})

print('[답변]', result['result'])
print('[출처]', result['source_documents'][0].page_content[:40])

[답변] 그에 대한 정보를 모르겠습니다. 경쟁사 D전자의 순이익에 대한 구체적인 데이터는 없습니다.
[출처] 기업의 매출과 영업이익이 개선될


In [29]:
# 실습 4 최근 많이 사용하는 Create_retrieval_chain 활용

# 1) LLM 준비
llm = ChatOpenAI(model='gpt-4o-mini')

# 2) 검색기 변환
retriever = db.as_retriever(search_kwargs={'k': 3})

# 3) 프롬프트 작성
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 참고하여 질문에 답하세요. /n/n {context}"),
    ("human", "{input}")])

# 4) 체인생성 : 검색된 문서를 LLM에 넣기
documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

# 5) 검색기와 문서를 연결
qa = create_retrieval_chain(retriever, documents_chain)

print('QA 챗봇 준비 완료')


QA 챗봇 준비 완료


In [34]:
# 여러 질문을 확인할 수 있는 변수와 for문 작성

questions = [
    '경제 뉴스의 주요 내용은?',
    '핵심 내용을 요약해서 설명해줘',
    '반도체에 대한 평가는 어때?',
    '금리에 대한 평가는 어때?'
]

for question in questions:
    result = qa.invoke({'input' : question})                         # 기존의 query
    print('[질문]', question)
    print('[답변]', result['answer'])                                # 기존의 result
    print('[출처]', result['context'][0].page_content[:40], '\n\n')  # 기존의 source_documents

[질문] 경제 뉴스의 주요 내용은?
[답변] 기사의 주요 내용은 반도체 수출 증가와 기준금리 동결에 관한 것입니다. 반도체 수출의 증가는 긍정적인 감성을 나타내고 있으며, 기준금리 동결은 중립적인 감성을 보입니다. 또한, 기사에서는 금리, 환율, 시장 변동성과 같은 핵심 키워드가 언급되고 있습니다.
[출처] 경제 뉴스 실습 자료
 금융 텍스트 분석 수업용 예시 기사 - 2026년 


[질문] 핵심 내용을 요약해서 설명해줘
[답변] 문서에서는 반도체 수출 증가에 대한 긍정적인 소식을 다루고 있습니다. 반도체 산업의 성장이 수출에 긍정적인 영향을 미치고 있으며, 이에 따라 경제 전반에도 긍정적인 변화가 기대되고 있습니다. 핵심 키워드는 '반도체 수출 증가', '긍정'입니다.
[출처] 위해 작성한 예시 자료입니다. 


[질문] 반도체에 대한 평가는 어때?
[답변] 반도체에 대한 평가는 긍정적입니다. 반도체 수출이 증가하고 있다는 점은 경제에 긍정적인 영향을 미치고 있으며, 이 분야의 성장 가능성이 높아 보입니다. 이러한 성장은 기술 발전과 함께 지속될 것으로 예상됩니다.
[출처] 조정을 검토하고 있지만 소비 위축 가능성 때문에 즉각적인 인상에는 신중한 


[질문] 금리에 대한 평가는 어때?
[답변] 현재 기준금리는 동결된 상태이며, 중립적인 입장을 유지하고 있습니다. 환율과 해외 금리의 조정을 검토하고 있지만 소비 위축 가능성으로 인해 금리를 즉각적으로 인상하는 것에는 신중한 모습입니다. 이는 금리에 대한 평가가 현 시점에서 안정성을 추구하고 있으며, 소비 경제에 미치는 영향을 고려하고 있다는 점에서 신중하다고 볼 수 있습니다.
[출처] 기준금리 동결
중립
금리, 환율, 시장 변동성 




## 3. 검색 품질 개선 전략

### 6. 검색 품질 개선 전략

- **chunk_size 조정**: 질문의 성격에 따라 짧게(정확한 사실) / 길게(맥락 설명) 조절하며 비교
- **overlap 조정**: 문장이 잘려 문맥이 끊기지 않도록 적절히 부여
- **출처 함께 제시**: 답변이 어느 조각에서 나왔는지 항상 확인 가능하게 하여 신뢰도 확보
- **모른다고 답하기**: 자료에 없는 질문에 그럴듯하게 지어내지(환각) 않고 "모른다"고 답하도록 프롬프트 설계

In [58]:
# 실습 1 Chunk_size를 바꿔가며 같은 질문의 검색 결과를 비교하시오

with open("report_v2.txt", 'r', encoding="utf-8") as f:
   report = f.read()

def search(size, query):
  sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=10)
  docs = sp.create_documents([report])
  vdb = Chroma.from_documents(docs, OpenAIEmbeddings(),
                              collection_name=f'chunk_{size}') # 청크 사이즈에 따라 문맥을 보완하는 코드
  return vdb.similarity_search(query, k=1)[0].page_content

q = '영업이익 증가 원인은?'

print('Chunk=30 : ', search(30,q)[:100])
print('Chunk=50 : ', search(50,q)[:100])
print('Chunk=100 : ', search(50,q)[:100])
print('Chunk=200 : ', search(200,q)[:100])

Chunk=30 :  영업이익은 10조원으로 전년 대비 25% 증가했다.
Chunk=50 :  영업이익은 10조원으로 전년 대비 25% 증가했다.
Chunk=100 :  영업이익은 10조원으로 전년 대비 25% 증가했다.
Chunk=200 :  B전자는 3분기 실적 발표에서 매출이 전년 대비 증가했다고 밝혔다.
3분기 연결 기준 매출은 70조원으로 전년 대비 12% 증가했다.
영업이익은 10조원으로 전년 대비 25% 증가


In [61]:
# 실습 2 Overlap(겹침) 유무에 따라 조각 경계가 어떻게 달라지는지 확인

for ov in [0,20]:
  sp = RecursiveCharacterTextSplitter(chunk_size=40, chunk_overlap=ov)
  d = sp.create_documents([report])
  print(f'overlap={ov} : 조각 {len(d)}개')
  print('첫 조각 끝 : ', d[0].page_content[-15:])

overlap=0 : 조각 33개
첫 조각 끝 :  년 대비 증가했다고 밝혔다.
overlap=20 : 조각 35개
첫 조각 끝 :  년 대비 증가했다고 밝혔다.


### 7. 키워드 검색 vs 의미 검색

| 구분 | 방식 | 특징 |
|---|---|---|
| 키워드 검색 | 문자열 포함 여부(`in`), BM25 | 정확한 단어가 있어야 찾아짐. 통계 기반 |
| 의미 검색 | 임베딩 + `similarity_search` | 정확한 단어가 없어도 의미가 비슷하면 찾아짐 |

- `BM25Retriever.from_texts(texts)`: 텍스트 리스트를 넣어 키워드 기반 검색기 생성
- 실무에서는 두 방식을 함께 쓰는 **하이브리드 검색**도 많이 활용됨

In [62]:
# 실습 3-1 키워드 검색(단어포함 여부)과 의미 검색(임베딩 유사도) 차이 비교

chunks = ['AI용 HBM 공급 확대가 기대된다',
          '메모리 수요 회복이 실적을 견인했다',
          '4분기 가이던스는 보수적으로 제시했다']

# 1) 키워드 검색 : 문자열에 'HBM'이 포함된 조각만 찾기
keyword_result = [c for c in chunks if 'HBM' in c]

# 2) Chunks를 Doucument로 변환
docs = [Document(page_content=c) for c in chunks]

# 3) Chunks를 기반으로 VectorDB생성
vdb = Chroma.from_documents(docs, OpenAIEmbeddings())

# 4) 의미 검색 : Chunks를 벡터DB에 넣고 '반도체'와 의미상 가장 가까운 조각 찾기
meaning_result = vdb.similarity_search('반도체', k=1)[0].page_content

print('[키워드] HBM : ', keyword_result)
print('[의미] 반도체 : ', meaning_result)

[키워드] HBM :  ['AI용 HBM 공급 확대가 기대된다']
[의미] 반도체 :  반도체 수요 회복이었다.


In [63]:
# 실습 3-2 키워드 검색(BM25)과의 의미 검색의 결과 차이를 비교하시오.

bm25 = BM25Retriever.from_texts(chunks)
bm25.k = 1 # 검색결과 중 가장 관련 높은 문서 1개만

q = 'HBM'

print('[키워드]', bm25.invoke(q)[0].page_content[:30])
print('[의미]', db.similarity_search(q, k=1)[0].page_content[:30])

[키워드] AI용 HBM 공급 확대가 기대된다
[의미] AI용 HBM 공급 확대가 기대된다


## 4. 금융 서비스 적용 고려사항

## 8. 금융 서비스 적용 시 고려사항

### 8-1. 비용 추정
- API는 입력/출력 토큰 수에 따라 과금됨 (예: gpt-4o-mini 기준 100만 토큰당 입력 $0.15, 출력 $0.60)
- 서비스 규모(일 요청 건수 × 건당 토큰 수)를 곱해 일/월 예상 비용을 미리 계산해봐야 함

### 8-2. 민감정보 마스킹
- 주민등록번호, 계좌번호 등 개인정보는 답변이나 로그에 그대로 노출되지 않도록 마스킹 필요
- 단순 문자열 슬라이싱(`[:6]`, `split('-')`) 또는 정규식(`re.sub`)으로 패턴을 찾아 치환
- 실제 서비스에서는 사용자 입력을 LLM에 보내기 **전에** 마스킹 처리하는 것이 안전

### 8-3. 면책 고지(Disclaimer)
- 금융 정보 챗봇은 "투자 권유가 아니다"라는 고지를 답변에 포함시켜야 법적/윤리적 리스크를 줄일 수 있음
- 시스템 프롬프트에 고지 문구를 명시하면 매 답변에 일관되게 포함시킬 수 있음

In [66]:
# 실습 1 입력 출력 토큰 수로 API 호출 비용을 추정하시오

# gpt-4o-mini 단가(100만 토큰당)
PRICE_IN, PRICE_OUT = 0.15, 0.60

def estimate(in_tok, out_tok):
  return (in_tok/1e6)*PRICE_IN + (out_tok/1e6)*PRICE_OUT

# 하루 1만건, 건당 입력 1000, 출력 300 토큰 가정

daily = estimate(1000,300) * 10000
print(f'일 예상 비용 : ${daily:.2f}')

일 예상 비용 : $3.30


In [67]:
# 실습 2-1 민감 정보(주민번호, 계좌)를 마스킹해 안전하게 처리하시오(문자열)

# 주민번호 마스킹 : 앞 6자리만 남기고 뒷자리는 *
jumin = '951225-1234567'
masked_jumin = jumin[:6] + '******'

# 계좌번호 마스킹 : 마지막 욲음만 *로 변경
account = '123-456-7890'
parts = account.split('-')       # ['123','456','7890']
parts[-1] = '*' * len(parts[-1]) # 마지막 묶음을 *로
masked_account = '-'.join(parts)

print(masked_jumin)
print(masked_account)

951225******
123-456-****


In [71]:
# 실습 2-2 민감정보(주민번호, 계좌)를 마스킹해 안전하게 처리하시오(정규식)

text = '고객 홍길동(951225-1234567), 계좌 123-456-78902'

# 주민번호 뒷자리 마스킹
text = re.sub(r'(\d{6})-\d{7}', r'\1-******',text) # (패턴, 치환할_문자열, 대상_문자열) \1 : 첫번째 그룹을 그대로 사용

# 계좌번호 마스킹
text = re.sub(r'(\d{3}-\d{3})-\d{5}', r'***-***-*****',text)

print(text)

고객 홍길동(951225-******), 계좌 ***-***-*****


In [77]:
# 실습 3 답변에 '투자 권유 아님' 고지를 포함하도록 시스템 프롬프트를 설정하시오

client = OpenAI()

system = '''너는 금융정보 도우미다. 답변끝에 반드시 다음 고지를 붙여라 : '본 내용은 투자 권유가 아니며 판단은 본인 책임입니다.'''

# 'content' : '반도체 전망은?' 사용
r = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': '반도체 전망은?'}
    ])

print(r.choices[0].message.content)

반도체 산업은 전세계적으로 지속적인 기술 발전과 수요 증가로 인해 긍정적인 전망을 보이고 있습니다. 특히 인공지능(AI), 5G 통신, 전기차, IoT(사물인터넷) 등과 같은 신기술의 발전이 반도체 수요를 더욱 촉진하고 있습니다. 

또한, 세계적인 공급망 재편이나 자국 생산 강화 정책이 반도체 산업에 영향을 미치고 있으며, 이러한 변화가 산업 구조에 긍정적인 변화를 가져올 가능성도 있습니다. 하지만, 반도체 산업의 특징적인 사이클성과 글로벌 시장의 변화에 따른 리스크도 존재하기 때문에 향후 전망은 주의 깊게 살펴봐야 합니다.

결론적으로, 반도체 산업은 중장기적으로 긍정적인 공급 전망이 있지만, 구체적인 투자 결정을 내리기 전에 시장 상황과 전문가 분석을 종합적으로 고려해야 합니다. 

본 내용은 투자 권유가 아니며 판단은 본인 책임입니다.


## 5. 미니 프로젝트

### 1. retriever 사용

In [79]:
# Step1. 데이터 준비

documents = [
    '''A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30% 증가했으며, 이는 서버용 메모리 가격 회복 덕분이다.
4분기에는 AI 가속기용 HBM 공급 확대가 실적 개선을 이끌 것으로 전망된다.
다만 중국 경쟁사의 저가 공세는 리스크 요인으로 지목된다.''',

    '''B바이오는 신약 후보물질 'BX-201'의 임상 3상에서 유효성을 확인했다고 발표했다.
회사는 내년 상반기 미국 식품의약국에 품목허가를 신청할 계획이다.
다만 임상 비용 증가로 3분기 영업손실은 전년 대비 확대됐다.
증권가는 허가 승인 여부에 따라 주가 변동성이 커질 수 있다고 분석했다.''',

    '''C자동차는 전기차 해외 판매량이 전년 대비 40% 증가했다고 밝혔다.
유럽과 동남아 시장에서 신모델 판매가 호조를 보였다.
반면 일부 차종에서 배터리 결함이 발견돼 자발적 리콜을 결정했다.
리콜 비용은 4분기 실적에 일부 반영될 전망이다.'''
]

print('문서 수 :', len(documents))
for i, doc in enumerate(documents, 1):
    print(f'\n[문서 {i}]')
    print(doc)

문서 수 : 3

[문서 1]
A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30% 증가했으며, 이는 서버용 메모리 가격 회복 덕분이다.
4분기에는 AI 가속기용 HBM 공급 확대가 실적 개선을 이끌 것으로 전망된다.
다만 중국 경쟁사의 저가 공세는 리스크 요인으로 지목된다.

[문서 2]
B바이오는 신약 후보물질 'BX-201'의 임상 3상에서 유효성을 확인했다고 발표했다.
회사는 내년 상반기 미국 식품의약국에 품목허가를 신청할 계획이다.
다만 임상 비용 증가로 3분기 영업손실은 전년 대비 확대됐다.
증권가는 허가 승인 여부에 따라 주가 변동성이 커질 수 있다고 분석했다.

[문서 3]
C자동차는 전기차 해외 판매량이 전년 대비 40% 증가했다고 밝혔다.
유럽과 동남아 시장에서 신모델 판매가 호조를 보였다.
반면 일부 차종에서 배터리 결함이 발견돼 자발적 리콜을 결정했다.
리콜 비용은 4분기 실적에 일부 반영될 전망이다.


In [80]:
# Step2. 문서 적재(임베딩 -> 벡터 DB)

# 1) 문서 조각내기
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20) # 진행도 조정
docs = splitter.create_documents(documents)

print('조각 수 :', len(docs))
for i, doc in enumerate(docs, 1):
    print(f'\n[{i}번째 조각]')
    print(doc.page_content)

# 2) 임베딩해 벡터 DB 생성 (기존 컬렉션 있으면 초기화)
try:
    db.delete_collection()
except:
    pass

db = Chroma.from_documents(docs, OpenAIEmbeddings())

print('\n저장된 조각 수 :', db._collection.count())

조각 수 : 6

[1번째 조각]
A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30% 증가했으며, 이는 서버용 메모리 가격 회복 덕분이다.

[2번째 조각]
4분기에는 AI 가속기용 HBM 공급 확대가 실적 개선을 이끌 것으로 전망된다.
다만 중국 경쟁사의 저가 공세는 리스크 요인으로 지목된다.

[3번째 조각]
B바이오는 신약 후보물질 'BX-201'의 임상 3상에서 유효성을 확인했다고 발표했다.
회사는 내년 상반기 미국 식품의약국에 품목허가를 신청할 계획이다.

[4번째 조각]
다만 임상 비용 증가로 3분기 영업손실은 전년 대비 확대됐다.
증권가는 허가 승인 여부에 따라 주가 변동성이 커질 수 있다고 분석했다.

[5번째 조각]
C자동차는 전기차 해외 판매량이 전년 대비 40% 증가했다고 밝혔다.
유럽과 동남아 시장에서 신모델 판매가 호조를 보였다.

[6번째 조각]
반면 일부 차종에서 배터리 결함이 발견돼 자발적 리콜을 결정했다.
리콜 비용은 4분기 실적에 일부 반영될 전망이다.

저장된 조각 수 : 6


In [81]:
# Step3. 검색기 만들기

# 1) 벡터DB를 검색기로 변환
retriever = db.as_retriever(search_kwargs={'k': 3})

print('검색기 준비 완료')

# 2) 테스트 질문으로 similarity_search가 잘 되는지 확인
test_query = '메모리 반도체 실적은 어때?'

hits = db.similarity_search(test_query, k=3)

print(f'\n[테스트 질문] {test_query}')
for i, h in enumerate(hits, 1):
    print(f'\n[{i}번째 검색 결과]')
    print(h.page_content)

검색기 준비 완료

[테스트 질문] 메모리 반도체 실적은 어때?

[1번째 검색 결과]
A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30% 증가했으며, 이는 서버용 메모리 가격 회복 덕분이다.

[2번째 검색 결과]
반면 일부 차종에서 배터리 결함이 발견돼 자발적 리콜을 결정했다.
리콜 비용은 4분기 실적에 일부 반영될 전망이다.

[3번째 검색 결과]
4분기에는 AI 가속기용 HBM 공급 확대가 실적 개선을 이끌 것으로 전망된다.
다만 중국 경쟁사의 저가 공세는 리스크 요인으로 지목된다.


In [83]:
# Step4. QA 챗봇 연결

qa = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model='gpt-4o-mini'),
    retriever=retriever,
    return_source_documents=True)

print('QA 챗봇 준비 완료')

# 테스트 질문
res = qa.invoke({'query': '메모리 반도체 실적은 어때?'})

print('\n[답변]', res['result'])
print('[출처]', res['source_documents'][0].page_content[:60])

QA 챗봇 준비 완료

[답변] A반도체의 3분기 매출은 45조원이었고, 영업이익은 8조원으로 전년 대비 30% 증가했습니다. 이는 서버용 메모리 가격 회복 덕분입니다. 4분기에는 AI 가속기용 HBM 공급 확대가 실적 개선을 이끌 것으로 전망되지만, 중국 경쟁사의 저가 공세가 리스크 요인으로 지목되고 있습니다.
[출처] A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30% 증가했으며, 이는


In [84]:
# Step5. 검증 & 개선

# 1) 자료에 '있는' 질문 3개로 정답을 잘 찾는지 확인
questions_in_doc = [
    'A반도체의 3분기 영업이익은 얼마야?',
    'B바이오 신약 임상 결과는 어때?',
    'C자동차 리콜 이유는 뭐야?'
]

print('=== 자료 안에 있는 질문 테스트 ===')
for q in questions_in_doc:
    res = qa.invoke({'query': q})
    print(f'\n[질문] {q}')
    print(f'[답변] {res["result"]}')
    print(f'[출처] {res["source_documents"][0].page_content[:50]}')

# 2) 자료에 '없는' 질문을 던져 '모른다'고 답하는지 확인
question_not_in_doc = 'D반도체의 최근 순이익은 얼마야?'

print('\n\n=== 자료에 없는 질문 테스트 ===')
res = qa.invoke({'query': question_not_in_doc})
print(f'[질문] {question_not_in_doc}')
print(f'[답변] {res["result"]}')

=== 자료 안에 있는 질문 테스트 ===

[질문] A반도체의 3분기 영업이익은 얼마야?
[답변] A반도체의 3분기 영업이익은 8조원입니다.
[출처] A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30%

[질문] B바이오 신약 임상 결과는 어때?
[답변] B바이오는 신약 후보물질 'BX-201'의 임상 3상에서 유효성을 확인했다고 발표했습니다.
[출처] B바이오는 신약 후보물질 'BX-201'의 임상 3상에서 유효성을 확인했다고 발표했다.
회

[질문] C자동차 리콜 이유는 뭐야?
[답변] C자동차의 리콜 이유는 일부 차종에서 배터리 결함이 발견되었기 때문입니다.
[출처] 반면 일부 차종에서 배터리 결함이 발견돼 자발적 리콜을 결정했다.
리콜 비용은 4분기 실적


=== 자료에 없는 질문 테스트 ===
[질문] D반도체의 최근 순이익은 얼마야?
[답변] 그에 대한 정보는 알 수 없습니다. D반도체의 최근 순이익에 대한 데이터가 제공되지 않았습니다.


In [89]:
# 3) chunk_size를 50 / 200으로 바꿔 검색 품질 비교

def build_and_search(size, query):
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20)
    d = sp.create_documents(documents)
    vdb = Chroma.from_documents(d, OpenAIEmbeddings(),
                                 collection_name=f'chunk_{size}')
    return vdb.similarity_search(query, k=1)[0].page_content

q = 'A반도체 실적 개선 이유는?'

print('Chunk=50  :', build_and_search(50, q))
print('Chunk=200 :', build_and_search(200, q))

Chunk=50  : A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
Chunk=200 : A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30% 증가했으며, 이는 서버용 메모리 가격 회복 덕분이다.
4분기에는 AI 가속기용 HBM 공급 확대가 실적 개선을 이끌 것으로 전망된다.
다만 중국 경쟁사의 저가 공세는 리스크 요인으로 지목된다.


### 2. create_retrieval_chain 사용

In [87]:
# Step4. QA 챗봇 연결 (create_retrieval_chain 사용)

# 1) LLM 준비
llm = ChatOpenAI(model='gpt-4o-mini')

# 2) 프롬프트 작성
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 참고하여 질문에 답하세요. 문서에 없는 내용이면 '자료에서 확인할 수 없습니다'라고 답하세요.\n\n{context}"),
    ("human", "{input}")
])

# 3) 문서결합 체인 생성 (검색된 문서를 LLM에 넣기)
documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

# 4) 검색기와 문서결합 체인을 연결
qa = create_retrieval_chain(retriever, documents_chain)

print('QA 챗봇 준비 완료')

# 테스트 질문
res = qa.invoke({'input': '메모리 반도체 실적은 어때?'})

print('\n[답변]', res['answer'])
print('[출처]', res['context'][0].page_content[:60])

QA 챗봇 준비 완료

[답변] 자료에서 확인할 수 없습니다.
[출처] A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30% 증가했으며, 이는


In [88]:
#  Step5. 검증 & 개선

# 1) 자료에 '있는' 질문 3개로 정답을 잘 찾는지 확인
questions_in_doc = [
    'A반도체의 3분기 영업이익은 얼마야?',
    'B바이오 신약 임상 결과는 어때?',
    'C자동차 리콜 이유는 뭐야?'
]

print('=== 자료 안에 있는 질문 테스트 ===')
for q in questions_in_doc:
    res = qa.invoke({'input': q})
    print(f'\n[질문] {q}')
    print(f'[답변] {res["answer"]}')
    print(f'[출처] {res["context"][0].page_content[:50]}')

# 2) 자료에 '없는' 질문을 던져 '모른다'고 답하는지 확인
question_not_in_doc = 'D반도체의 최근 순이익은 얼마야?'

print('\n\n=== 자료에 없는 질문 테스트 ===')
res = qa.invoke({'input': question_not_in_doc})
print(f'[질문] {question_not_in_doc}')
print(f'[답변] {res["answer"]}')

=== 자료 안에 있는 질문 테스트 ===

[질문] A반도체의 3분기 영업이익은 얼마야?
[답변] A반도체의 3분기 영업이익은 8조원입니다.
[출처] A반도체는 3분기 매출 45조원, 영업이익 8조원을 기록했다.
전년 대비 영업이익이 30%

[질문] B바이오 신약 임상 결과는 어때?
[답변] B바이오는 신약 후보물질 'BX-201'의 임상 3상에서 유효성을 확인했다고 발표했습니다.
[출처] B바이오는 신약 후보물질 'BX-201'의 임상 3상에서 유효성을 확인했다고 발표했다.
회

[질문] C자동차 리콜 이유는 뭐야?
[답변] 자료에서 확인할 수 없습니다.
[출처] 반면 일부 차종에서 배터리 결함이 발견돼 자발적 리콜을 결정했다.
리콜 비용은 4분기 실적


=== 자료에 없는 질문 테스트 ===
[질문] D반도체의 최근 순이익은 얼마야?
[답변] 자료에서 확인할 수 없습니다.
